# 🏥 Medical Report Generation — Transformer (Swin + BioGPT)

## 🗺️ Hibrit Eğitim Stratejisi

| Aşama | Nerede | Epoch | Durum |
|-------|--------|-------|-------|
| **Faz 1** | 🖥️ Local PC | 1 – 2 | Encoder **dondurulmuş** → hızlı (~1 saat) |
| **Faz 2** | ☁️ Kaggle GPU | 3 – 20 | Encoder **açık** (fine-tune) → büyük VRAM gerekli |

> **Akış:** Local'de epoch 1-2'yi eğit → `best_model/` klasörünü Kaggle'a yükle → bu notebook epoch 3'ten devam eder.

Repo: https://github.com/MustafaYagizKaragoz/medical-report-generation-cxr

---
## 🖥️ FAZ 1 — LOCAL PC'DE ÇALIŞTIR (epoch 1-2)
> **Bu hücreleri Kaggle'da çalıştırma!** Sadece local PC için.

In [ ]:
# ============================================================
# 🖥️ LOCAL PC — Epoch 1-2 (Encoder dondurulmuş)
# Bu hücreyi sadece LOCAL makinende çalıştır!
# ============================================================
import os, sys

LOCAL_REPO = r'C:\Users\ygz70\Desktop\Bitirme Projesi Yeni'
sys.path.insert(0, LOCAL_REPO)
os.chdir(LOCAL_REPO)

from config import Config

# ── Local path'ler ──────────────────────────────────────────
Config.TRAIN_PROCESSED_CSV = os.path.join(Config.DATA_DIR, 'train_processed.csv')
Config.VAL_PROCESSED_CSV   = os.path.join(Config.DATA_DIR, 'val_processed.csv')

# ── Local eğitim ayarları (küçük batch, encoder frozen) ─────
Config.TRANSFORMER_BATCH_SIZE        = 1     # Local VRAM için
Config.TRANSFORMER_GRAD_ACCUM_STEPS  = 16    # Efektif batch = 16
Config.TRANSFORMER_USE_AMP           = True
Config.TRANSFORMER_MAX_LENGTH        = 128
Config.TRANSFORMER_IMAGE_SIZE        = 384
Config.TRANSFORMER_LEARNING_RATE     = 5e-5
Config.NUM_WORKERS                   = 0     # Windows: 0
Config.EPOCHS                        = 2     # ← SADECE 2 EPOCH!
Config.PATIENCE                      = 99    # Early stopping kapalı
Config.TRANSFORMER_FINE_TUNE_START_EPOCH = 999  # Fine-tune YOK (local'de)
Config.TRANSFORMER_RESUME            = False

# Checkpoint local klasöre kaydedilsin
Config.TRANSFORMER_CHECKPOINT_DIR    = os.path.join(LOCAL_REPO, 'checkpoints_transformer')
Config.TRANSFORMER_CHECKPOINT_FILE   = os.path.join(Config.TRANSFORMER_CHECKPOINT_DIR, 'best_model')

Config.setup()
print('✅ Local config yüklendi!')
print(f'   Checkpoint dizini: {Config.TRANSFORMER_CHECKPOINT_DIR}')
print(f'   Epoch sayısı: {Config.EPOCHS}')
print('⚠️  Eğitim bittikten sonra checkpoints_transformer/best_model/ klasörünü Kaggle Dataset olarak yükle!')

In [ ]:
# 🖥️ LOCAL — Eğitimi başlat (epoch 1-2)
import train_transformer
train_transformer.main()
print('\n✅ Faz 1 bitti! checkpoints_transformer/best_model/ klasörünü Kaggle Dataset olarak yükle.')

---
## ☁️ FAZ 2 — KAGGLE'DA ÇALIŞTIR (epoch 3-20, encoder açık)

### ⚠️ Kaggle'a Geçmeden Önce Yapman Gerekenler:
1. Local'de epoch 1-2 eğitimi bitti
2. `checkpoints_transformer/best_model/` klasörünü **Kaggle Dataset** olarak yükle
   - Kaggle → Datasets → New Dataset → klasörü zip'le yükle
   - Dataset adı örnek: `transformer-checkpoint-ep2`
3. Bu notebook'a **"+Add Data"** ile ekle
4. Aşağıdaki `CHECKPOINT_DATASET_PATH` değişkenini güncelle

In [ ]:
# GPU kontrolü
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# Gerekli paketleri yükle
!pip install -q transformers>=4.30.0 rouge-score pycocoevalcap

In [ ]:
import os

# ── Repoyu klonla ───────────────────────────────────────────
if not os.path.exists('/kaggle/working/repo'):
    !git clone https://github.com/MustafaYagizKaragoz/medical-report-generation-cxr.git /kaggle/working/repo
else:
    # Varsa güncelle
    !git -C /kaggle/working/repo pull

os.chdir('/kaggle/working/repo')
print('✅ Repo hazır:', os.getcwd())

In [ ]:
# ── Veri yolları ────────────────────────────────────────────
# Kaggle'a yüklediğin dataset adına göre güncelle:
TRAIN_CSV  = '/kaggle/input/mimic-cxr-processed/train_processed.csv'
VAL_CSV    = '/kaggle/input/mimic-cxr-processed/val_processed.csv'
TEST_CSV   = '/kaggle/input/mimic-cxr-processed/test_processed.csv'
IMAGE_DIR  = '/kaggle/input/mimic-cxr-images/files'

# ⬇️ Local'den yüklediğin checkpoint dataset'in yolu:
# (Kaggle Dataset adını kendi yüklediğin isimle değiştir)
CHECKPOINT_DATASET_PATH = '/kaggle/input/transformer-checkpoint-ep2/best_model'

# Kontrol
for label, path in [('Train CSV', TRAIN_CSV), ('Val CSV', VAL_CSV),
                     ('Image Dir', IMAGE_DIR), ('Checkpoint', CHECKPOINT_DATASET_PATH)]:
    exists = os.path.exists(path)
    print(f"{'✅' if exists else '❌'} {label}: {path}")

In [ ]:
# ── Kaggle Config ───────────────────────────────────────────
import sys
sys.path.insert(0, '/kaggle/working/repo')

from config import Config

# Veri yolları
Config.TRAIN_PROCESSED_CSV = TRAIN_CSV
Config.VAL_PROCESSED_CSV   = VAL_CSV
Config.TEST_PROCESSED_CSV  = TEST_CSV
Config.IMAGE_DIR           = IMAGE_DIR

# Checkpoint: Local'den yüklenen epoch-2 checkpoint
Config.TRANSFORMER_RESUME              = True
Config.TRANSFORMER_CHECKPOINT_FILE     = CHECKPOINT_DATASET_PATH
Config.TRANSFORMER_CHECKPOINT_DIR      = '/kaggle/working/checkpoints_transformer'
Config.LOG_DIR                         = '/kaggle/working/logs'

# Kaggle GPU (P100/T4 16GB) ayarları — encoder AÇIK olacak
Config.TRANSFORMER_BATCH_SIZE           = 2    # encoder açık → biraz daha bellek
Config.TRANSFORMER_GRAD_ACCUM_STEPS     = 8    # Efektif batch = 2×8 = 16
Config.TRANSFORMER_USE_AMP              = True
Config.TRANSFORMER_MAX_LENGTH           = 128
Config.TRANSFORMER_IMAGE_SIZE           = 384
Config.TRANSFORMER_LEARNING_RATE        = 5e-5
Config.NUM_WORKERS                      = 2
Config.EPOCHS                           = 20   # Epoch 3'ten 20'ye kadar
Config.PATIENCE                         = 5

# ⭐ KRITIK: Fine-tune epoch=3 olarak ayarla (Kaggle'da anında başlasın)
# start_epoch=3 olacak, fine-tune check: epoch==3 → unfreeze!
Config.TRANSFORMER_FINE_TUNE_START_EPOCH = 3

Config.setup()

print('✅ Kaggle Config hazır!')
print(f'   Resume checkpoint : {Config.TRANSFORMER_CHECKPOINT_FILE}')
print(f'   Fine-tune start   : Epoch {Config.TRANSFORMER_FINE_TUNE_START_EPOCH}')
print(f'   Batch size        : {Config.TRANSFORMER_BATCH_SIZE} (grad_accum={Config.TRANSFORMER_GRAD_ACCUM_STEPS})')
print(f'   Efektif batch     : {Config.TRANSFORMER_BATCH_SIZE * Config.TRANSFORMER_GRAD_ACCUM_STEPS}')

In [ ]:
# GPU ve torch kontrolü
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ☁️ KAGGLE — Eğitimi epoch 3'ten devam ettir (encoder açık)
#
# train_transformer.py'nin main() fonksiyonu:
#   1. TRANSFORMER_RESUME=True → epoch-2 checkpoint'ini yükler
#   2. start_epoch=3 olur (epoch 2'den sonrası)
#   3. TRANSFORMER_FINE_TUNE_START_EPOCH=3 → epoch 3 girince encoder açılır
#   4. Differential LR: encoder=5e-6, decoder=5e-5
#
import train_transformer
train_transformer.main()

In [ ]:
# En iyi modeli zip'le ve indir
import shutil

shutil.make_archive(
    '/kaggle/working/best_model_transformer',
    'zip',
    '/kaggle/working/checkpoints_transformer/best_model'
)
print('✅ Model zip olarak kaydedildi: /kaggle/working/best_model_transformer.zip')
print('   Sağ taraftaki Output sekmesinden indirebilirsin.')